In [1]:
'''File Description

## Author: Yixuan GUO

## Purpose:
This script is designed to detect and output zero-time points at different detection time windows. The detection time windows include ±[1, 2, 4, 6, 12, 18, 24, 30, 36] hours. By running all the code in this file, users will obtain a list of 76 detected zero-time points and the ±720 hours of effective spreads

## Functionality:
- Detects zero-time points based on each detection windows
- Outputs the results in a structured format (panel) for further analysis

## How to Use:
1. Download the entire folder and retain the folder name and structure
3. Run the code locally without modifying
4. The script will automatically process the data and generate the results

## Output:
- The script outputs **76 zero-time points**, detected across different detection windows.
- Results are saved in the local folder where the script is executed.
- Each detection window (e.g., ±1 hour, ±2 hours) is analyzed, and the results are consolidated into a single output file or multiple files, as defined in the code.

## Notes:
- Ensure the input data (if applicable) is correctly formatted and placed in the same folder as the script.
- For any issues or questions, please contact the author or refer to the documentation (if provided).

## Example:
When the script is executed, the output might look like:

'''

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

'''
please download the entire folder and run the code locally
'''
event_filter_data = "02_event_filter.csv"
btc_price_data = "03_BTC-USD_price.csv"
spread_raw_data = "04_hourly_avg_e_spread.csv"
coin_spread = "BTC"


In [2]:
def load_data(btc_price_data, spread_raw_data, event_filter_data, coin_spread):

    price_data = pd.read_csv(btc_price_data, parse_dates=['datetime'])          # parse means to convert the date column to datetime
    price_data = price_data.sort_values(by='datetime')                          # sort the data by date

    spread_data = pd.read_csv(spread_raw_data, parse_dates=['datetime'])    # parse means to convert the date column to datetime
    spread_data = spread_data.sort_values(by='datetime')                    # sort the data by date
    spread_data[coin_spread] = pd.to_numeric(spread_data[coin_spread], errors='coerce')
    spread_data[coin_spread] = spread_data[coin_spread].fillna(0)

    event_data = pd.read_csv(event_filter_data, parse_dates=['date'])

    return price_data, spread_data, event_data


def detect_event_hour(price_data, spread_data, event_date, n_hours):

    event_day_price = price_data[    # Filter the data for the event day, including previous and next days
        (price_data['datetime'] >= pd.Timestamp(event_date) - pd.Timedelta(hours=n_hours)) &    # include data of previous n_days
        (price_data['datetime'] < pd.Timestamp(event_date) + pd.Timedelta(hours=n_hours))       # include data of next n_days
    ]
    event_day_price['high_low_diff'] = abs(event_day_price['high'] - event_day_price['low'])    # calculate the difference between high and low for each hour
    event_ob_hour = event_day_price.loc[event_day_price['high_low_diff'].idxmax(), 'datetime']  # take the hour with the maximum difference as the event_hour
    event_ob_spread = spread_data.loc[spread_data['datetime'] == event_ob_hour, coin_spread].values[0]  # retrieve the spread value at the event_ob_hour

    return event_ob_hour, event_ob_spread


def calculate_baseline(spread_data, event_ob_hour, m_days):

    baseline_data = spread_data[    # Filter the data for the baseline period
        (spread_data['datetime'] >= pd.Timestamp(event_ob_hour) - pd.Timedelta(hours=24*m_days)) &  # include data btw m_days
        (spread_data['datetime'] < pd.Timestamp(event_ob_hour))                                     # and 1 hour before the event_ob_hour
    ]
    baseline_avg_spread = baseline_data[coin_spread].mean()   # Calculate the average spread

    return baseline_data, baseline_avg_spread


def calculate_diff(spread_data, event_ob_hour, baseline_avg_spread, k_days):

    post_event_data = spread_data[    # Filter the data for the post-event period
        (spread_data['datetime'] >= pd.Timestamp(event_ob_hour)) &   # include date between event_ob_hour
        (spread_data['datetime'] < pd.Timestamp(event_ob_hour) + pd.Timedelta(hours=24*k_days))  # and k_days afterward
    ]
    post_event_data['diff_spread'] = post_event_data[coin_spread] - baseline_avg_spread  # Calculate the difference for each hour

    return post_event_data

In [3]:
def generate_panel_df(price_data, spread_data, event_data, n_hours):

    # Initialize panel_df
    num_rows = m_days * 24 + 1 + k_days * 24   # no. of rows: baseline + event + post-event
    num_cols = len(event_data)                 # no. of columns: number of events
    panel_df = pd.DataFrame(index=range(num_rows), columns=[f"event{i+1}" for i in range(num_cols)])   # Create the empty DataFrame
    panel_df[:] = np.nan                       # Initialize with NaN

    # Iterate over each event
    for i, event_date in enumerate(event_data['date']):
        # Calculate baseline, event hour, and post-event differences
        event_ob_hour, event_ob_spread = detect_event_hour(price_data, spread_data, event_date, n_hours)
        baseline_data, baseline_avg_spread = calculate_baseline(spread_data, event_ob_hour, m_days)
        baseline_spread_array = np.array(baseline_data[coin_spread]) # Convert the spread data to an array
        post_event_data = calculate_diff(spread_data, event_ob_hour, baseline_avg_spread, k_days)
        diff_spread_array = np.array(post_event_data[coin_spread])   # Convert the spread data to an array

        event_array = np.concatenate([            # Combine baseline, event, and post-event data
            baseline_spread_array[:24 * m_days],  # Baseline hours
            np.array([event_ob_spread]),          # Event observation hour
            diff_spread_array[:24 * k_days]       # Post-event hours
        ])
        event_array = event_array.reshape(-1)     # Ensure vertical structure

        column_name = f"event{i+1}"               # Name the column with the event number
        panel_df.loc[:len(event_array) - 1, column_name] = event_array    # Insert the event array into the DataFrame

    # Insert a new column at the beginning of the DataFrame, representing the hour timestamps
    panel_df.insert(0, 'Time', list(range(-m_days * 24, 0)) + [0] + list(range(1, k_days * 24 + 1)))

    # Save the DataFrame as a CSV file
    panel_df.to_csv(f'panels_different_time_windows/panel_df_{int(n_h)}_hours.csv', index=False)
    print(f"Panel DataFrame saved to panel_df_{int(n_h)}_hours.csv.")

    return panel_df


In [7]:
n_hours = 18   # DETERMINED: the event_ob_hour in a range of n_h hours before and after the event outbreak
m_days = 30    # DETERMINED: how many days for calculating the baseline
k_days = 10    # DETERMINED: how many days for calculating the difference
coin_spread = "BTC"

# load
price_data, spread_data, event_data = load_data(btc_price_data, spread_raw_data, event_filter_data, coin_spread)
event_ob_hours = []

# iterate each event date
for i, event_date in enumerate(event_data['date']):
    event_ob_hour, event_ob_spread = detect_event_hour(price_data, spread_data, event_date, n_hours)
    event_ob_hours.append({'event_date': event_date, 'event_ob_hour': event_ob_hour})

# Add event_index column
event_ob_hour_df = pd.DataFrame(event_ob_hours)
event_ob_hour_df['event_index'] = [f'event{i+1}' for i in range(len(event_ob_hour_df))]

# save to csv
event_ob_hour_df.set_index('event_index', inplace=True)
event_ob_hour_df.to_csv('04_[Outcome]_event_ob_hour.csv', index=True)